### Before Running Code

- Make sure to have your **SLEAP files** in `.h5` format and in their own directory.
- It’s recommended to use **VS Code** to run the notebooks. You need the `keypoint-moseq` environment active in VS Code. Here's how:

    - **For Windows**: Press `Ctrl + Shift + P`, then select **Python: Select Interpreter**. Choose your `keypoint` environment from the list.
    
    - **For macOS**: Press `Cmd + Shift + P`, then follow the same steps as Windows.

#### Step 1: 
**Interpolate your `.h5` files** to remove any `NaN` values. Save the cleaned files to any directory of your choice. This ensures downstream analyses won't break due to missing data.

In [ ]:
# Required imports for Entire Flow
import keypoint_moseq as kpms
import os
import glob
import h5py
import numpy as np
from scipy.interpolate import interp1d

In [ ]:
# Interpolation Function
def fill_missing(Y, kind="linear"):
    """Fills missing values independently along each dimension after the first."""
    initial_shape = Y.shape
    Y = Y.reshape((initial_shape[0], -1))

    for i in range(Y.shape[-1]):
        y = Y[:, i]
        x = np.flatnonzero(~np.isnan(y))
        if len(x) < 2:
            continue  # Skip if not enough points to interpolate

        f = interp1d(x, y[x], kind=kind, fill_value=np.nan, bounds_error=False)
        xq = np.flatnonzero(np.isnan(y))
        y[xq] = f(xq)

        mask = np.isnan(y)
        if np.any(~mask):
            y[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), y[~mask])

        Y[:, i] = y

    return Y.reshape(initial_shape)

In [ ]:
# Apply interpolation code to each h5 file in your input folder and save new files to 
# your output file. 

input_folder = r"input folder path"
output_folder = r"output folder path"
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.endswith(".h5"):
        filepath = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)
        
        with h5py.File(filepath, "r") as f:
            dset_names = list(f.keys())
            locations = f["tracks"][:].T
            node_names = [n.decode() for n in f["node_names"][:]]

            print(f"Processing file: {filepath}")
            print(f"Datasets: {dset_names}")
            print(f"Locations shape: {locations.shape}")
            print(f"Nodes: {node_names}")

            # Interpolate
            data_interp = fill_missing(locations)

            # Save interpolated version
            with h5py.File(output_path, "w") as f_out:
                # Copy all datasets and attributes
                for key in f.keys():
                    if key == "tracks":
                        # Replace the "tracks" dataset with interpolated data
                        f_out.create_dataset("tracks", data=data_interp.T)
                    else:
                        # Copy other datasets as is
                        f.copy(key, f_out)
                # Copy attributes from the root level
                for attr_name, attr_value in f.attrs.items():
                    f_out.attrs[attr_name] = attr_value
    print(np.isnan(data_interp).any())
    print(f"Interpolated and saved: {output_path}")

False
Interpolated and saved: Z:\2024-09_FoodDep_Amanda\SLEAP\female\h5 (by only)\interpolated\20240919_FoodDep_F_postCAP_2108_2109_bottom.analysis.h5
Processing file: Z:\2024-09_FoodDep_Amanda\SLEAP\female\h5 (by only)\20240919_FoodDep_F_postCAP_2108_2109_bottom.analysis.h5
Datasets: ['edge_inds', 'edge_names', 'instance_scores', 'labels_path', 'node_names', 'point_scores', 'provenance', 'track_names', 'track_occupancy', 'tracking_scores', 'tracks', 'video_ind', 'video_path']
Locations shape: (54000, 8, 2, 1)
Nodes: ['nose', 'R_frontpaw', 'L_frontpaw', 'R_hindpaw', 'L_hindpaw', 'tail_base', 'tail_end', 'body_center']
False
Interpolated and saved: Z:\2024-09_FoodDep_Amanda\SLEAP\female\h5 (by only)\interpolated\20240919_FoodDep_F_postCAP_2108_2109_bottom.analysis.h5
Processing file: Z:\2024-09_FoodDep_Amanda\SLEAP\female\h5 (by only)\20240919_FoodDep_F_postCAP_2112_2113_bottom.analysis.h5
Datasets: ['edge_inds', 'edge_names', 'instance_scores', 'labels_path', 'node_names', 'point_score

#### Step 2:

Apply the entire KPMS analysis on your new output folder of the interpolated h5 files.

In [ ]:
# Setting interpolated h5 file folder
h5_folder = r"interpolated folder path"
print("Folder path type: ", type(h5_folder))

In [ ]:
# Check if all files loaded
h5_files = glob.glob(os.path.join(h5_folder, "*analysis.h5"))
print(f"Found {len(h5_files)} HDF5 files.")

In [ ]:
# Set your project directory. This is not the h5 folder path.
# This should be a folder with all your project files.
directory = r"project directory path"

In [ ]:
# Create a configuration file
kpms.io.generate_config(directory)
print("Config file generated")
config_path = os.path.join(directory, "config.yml")

if os.path.exists(config_path):
    config = lambda: kpms.load_config(directory)
else:
    raise FileNotFoundError(f"Config file not found at {config_path}. Please generate one.")

In [ ]:
# Load keypoints
coordinates, confidences, bodyparts = kpms.load_keypoints(h5_folder, 'sleap')
print("Shape of coordinates:", {k: v.shape for k, v in coordinates.items()})
print("Bodyparts:", bodyparts)

In [ ]:
# Edit configuration file to include bodyparts. 
print("Config contents: ", config())
config_data = config()
config_data["bodyparts"] = ['nose', 'R_frontpaw', 'L_frontpaw', 'R_hindpaw', 'L_hindpaw', 'tail_base', 'tail_end', 'body_center']
config_data["use_bodyparts"] = ['nose', 'R_frontpaw', 'L_frontpaw', 'R_hindpaw', 'L_hindpaw', 'tail_base', 'tail_end', 'body_center']
config_data["skeleton"] = [['nose', 'R_frontpaw'], ['R_frontpaw', 'L_frontpaw'], ['L_frontpaw', 'R_hindpaw'], ['R_hindpaw', 'L_hindpaw']]
config_data["anterior_bodyparts"] = ['nose']
config_data["posterior_bodyparts"] = ['body_center']

In [ ]:
# Format data 
data, metadata = kpms.format_data(coordinates, confidences, **config_data)
print("Data formatted successfully.")

In [ ]:
# Fit PCA
pca = kpms.fit_pca(**data, **config())
kpms.save_pca(pca, directory)

kpms.print_dims_to_explain_variance(pca, 0.9)
kpms.plot_scree(pca, project_dir=directory)
kpms.plot_pcs(pca, project_dir=directory, **config_data)

In [ ]:
kpms.update_config(directory, latent_dim=4)

In [ ]:
# Initializing the model
model = kpms.init_model(data, pca=pca, **config_data)

In [ ]:
# Apply AR-HMM to Data 
num_ar_iters = 50

In [ ]:
# the below code might take a few minutes to an hour to process based on iteration number
model, model_name = kpms.fit_model(
    model, data, metadata, directory, ar_only=True, num_iters=num_ar_iters
)

**Note** : recommend not running the previous blocks again unless you want to create a new model. 

This is the best checkpoint if you want to apply your model to different files.

In [ ]:
# Retrieve model name for future running
print(model_name)
mn = model_name

In [ ]:
# Path to checkpoint file saved in your project directory
checkpoint_path = r"checkpoint path"

In [ ]:
# See all iterations from your checkpoint file
with h5py.File(checkpoint_path, "r") as f:
    print("Available iterations:", list(f["model_snapshots"].keys()))

In [ ]:
# Error checking if checkpoint has all 3 keys: data, metadata, model_snapshots
with h5py.File(checkpoint_path, "r") as f:
    print("Available keys:", list(f.keys()))  # Check available top-level keys

In [ ]:
# Read data contents
with h5py.File(checkpoint_path, "r") as f:
    try:
        print("Contents of 'data':", list(f["data"].keys()))  # List all datasets in "data"
    except Exception as e:
        print(f"Error reading 'data': {e}")

In [ ]:
# Error checking if data is too large a file
with h5py.File(checkpoint_path, "r") as f:
    data_group = f["data"]
    for key in data_group.keys():
        dataset = data_group[key]
        print(f"Dataset: {key}, Shape: {dataset.shape}, Type: {dataset.dtype}")

In [ ]:
# Error checking if permissions for writing and reading file
print("File exists:", os.path.exists(checkpoint_path))
print("Readable:", os.access(checkpoint_path, os.R_OK))
print("Writable:", os.access(checkpoint_path, os.W_OK))

In [ ]:

# load model checkpoint from the last iteration
model, data, metadata, current_iter = kpms.load_checkpoint(
    directory, mn, iteration=50
)

# modify kappa to maintain the desired syllable time-scale (short behavior lengths = larger kappa and long behavior lengths = short kappa values.)
model = kpms.update_hypparams(model, kappa=1000)

# run fitting for an additional 500 iters (decrease if you want to decrease runtime). This typically takes a few hours.
model = kpms.fit_model(
    model,
    data,
    metadata,
    directory,
    mn,
    ar_only=False,
    start_iter=current_iter,
    num_iters=current_iter + 100,
)[0]

In [ ]:
# modify a saved checkpoint so syllables are ordered by frequency
kpms.reindex_syllables_in_checkpoint(directory, model_name)

In [ ]:
# load the most recent model checkpoint
model, data, metadata, current_iter = kpms.load_checkpoint(directory, model_name)

# extract results
results = kpms.extract_results(model, metadata, directory, model_name)

In [ ]:
# Save results as a csv file
kpms.save_results_as_csv(results, directory, model_name)

#### Step 4: 

Visualize your results

In [ ]:
# Trajectory Plots
results = kpms.load_results(directory, model_name)
kpms.generate_trajectory_plots(coordinates, results, directory, model_name, **config_data)

In [ ]:
# Dendogram
kpms.plot_similarity_dendrogram(coordinates, results, directory, model_name, **config_data)

In [ ]:
# Grid movies for each syllable
# Videos need specific file name sometimes so check error messages for what it should be called.
kpms.generate_grid_movies(results, directory, model_name, coordinates=coordinates, **config_data)

In [ ]:
# Label your syllables
kpms.interactive_group_setting(directory, mn)

In [ ]:
moseq_df = kpms.compute_moseq_df(directory, mn, smooth_heading=True)
moseq_df

In [ ]:
stats_df = kpms.compute_stats_df(
    directory,
    mn,
    moseq_df,
    min_frequency=0.005,  # threshold frequency for including a syllable in the dataframe
    groupby=["group", "name"],  # column(s) to group the dataframe by
    fps=30,
)  # frame rate of the video from which keypoints were inferred

stats_df

In [ ]:
# save moseq_df
save_dir = os.path.join(directory, mn) # directory to save the moseq_df dataframe
moseq_df.to_csv(os.path.join(save_dir, 'moseq_df.csv'), index=False)
print('Saved `moseq_df` dataframe to', save_dir)

# save stats_df
save_dir = os.path.join(directory, mn)
stats_df.to_csv(os.path.join(save_dir, 'stats_df'), index=False)
print('Saved `stats_df` dataframe to', save_dir)

In [ ]:
kpms.label_syllables(directory, mn, moseq_df)

In [ ]:
kpms.plot_syll_stats_with_sem(
    stats_df,
    directory,
    mn,
    plot_sig=True,  # whether to mark statistical significance with a star
    thresh=0.05,  # significance threshold
    stat="frequency",  # statistic to be plotted (e.g. 'duration' or 'velocity_px_s_mean')
    order="stat",  # order syllables by overall frequency ("stat") or degree of difference ("diff")
    ctrl_group="a",  # name of the control group for statistical testing
    exp_group="b",  # name of the experimental group for statistical testing
    figsize=(8, 4),  # figure size
    groups=stats_df["group"].unique(),  # groups to be plotted
)

In [ ]:
normalize = "bigram"  # normalization method ("bigram", "rows" or "columns")

trans_mats, usages, groups, syll_include = kpms.generate_transition_matrices(
    directory,
    mn,
    normalize=normalize,
    min_frequency=0.005,  # minimum syllable frequency to include
)

kpms.visualize_transition_bigram(
    directory,
    mn,
    groups,
    trans_mats,
    syll_include,
    normalize=normalize,
    show_syllable_names=True,  # label syllables by index (False) or index and name (True)
)

In [ ]:
# Generate a transition graph for each single group

kpms.plot_transition_graph_group(
    directory,
    mn,
    groups,
    trans_mats,
    usages,
    syll_include,
    layout="circular",  # transition graph layout ("circular" or "spring")
    show_syllable_names=False,  # label syllables by index (False) or index and name (True)
)

In [ ]:
# Generate a difference-graph for each pair of groups.

kpms.plot_transition_graph_difference(
    directory, mn, groups, trans_mats, usages, syll_include, layout="circular"
)  # transition graph layout ("circular" or "spring")